In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from transformers import get_cosine_schedule_with_warmup
import os
from torch.utils.data import Dataset, DataLoader

from dataset_loaders import Train_img_Dataset
# from models.NKD_linear_layer import NKD_latent_layers, NKD_latent_layers_v2, save_checkpoint, load_checkpoint
# from NKD_kenel_layer import NKD_kernel_layers_v1
# from NKD_test import Test_denoisers


/opt/anaconda3/envs/nkd-aditya/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def train(args, verbose = False):
    # Setup device
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    trainset = Train_img_Dataset('./data/img_train', args.color_mode, args.sigma_min, args.sigma_max, 
                                 patch_size=args.patch_size, batch_size=args.imgs_per_epoch, 
                                 req_patches=args.req_patches, factor=args.factor, no_patches=args.no_patches)
    train_loader = DataLoader(
                trainset,
                batch_size=1,  # Process one full image at a time
                shuffle=False,
                pin_memory=True
            )
    
    # Create model
    in_nc = out_nc = 1 if args.color_mode == 'gray' else 3
    model = args.architecture(args, device).to(device)

    _state = f'_{str(model)}{args.sigma_min}_{args.sigma_max}'
    args.out_folder = args.out_folder+_state
    # Check if the output folder exists
    if not os.path.exists(args.out_folder):
        # Create the folder
        os.makedirs(args.out_folder)
        os.makedirs(os.path.join(args.out_folder, 'ckpt'))
        os.makedirs(os.path.join(args.out_folder, 'imgs'))
        if verbose:
            print(f"Folder created at: {args.out_folder}")

    # Setup optimizer, scheduler and criterion
    optimizer = optim.AdamW(model.parameters(), lr=args.lr)
    if args.sch == 'multistep':
        if isinstance(args.lr_step, str):
            milestones = [int(x) for x in args.lr_step.split(',')]
        else:
            milestones = [args.lr_step]
        # Multi-step learning rate scheduler
        scheduler = optim.lr_scheduler.MultiStepLR(
            optimizer, milestones=milestones, gamma=args.lr_gamma
        )
    elif args.sch == 'cosine':
        # Cosine annealing learning rate scheduler
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=args.epochs
        )
    elif args.sch == 'warmup_cosine':
        scheduler = get_cosine_schedule_with_warmup(
            optimizer=optimizer,
            num_warmup_steps=args.num_warmup_steps,
            num_training_steps=args.epochs,
            num_cycles=args.num_cycles
        )
    else:
        raise ValueError(f"scehduler(={args.sch}) is not defined")
    criterion = nn.MSELoss()
    # For tracking best validation results
    best_psnr = 0
    best_ssim = 0

    # Training loop
    total_steps = 0
    model.train()

    losses = []
    val_metric = []

    # try:
    model.train()
    for epoch in range(args.epochs):
        batch_count = 0
        current_lr = optimizer.param_groups[0]['lr']

        loss = model.train_step(train_loader, criterion, optimizer, lamda=args.l1_reg_lamda)
        total_steps += 1
        batch_count += 1
        losses.append(loss)

        # Print progress
        if total_steps % args.progress_freq == 0:
            print(f"Epoch {epoch+1}/{args.epochs} | "
                    f"Loss: {loss:.6f} | lr = {current_lr}", end='\r')

        # Save epoch loss to log
        with open(os.path.join(args.out_folder, 'training_log.txt'), 'a') as f:
            f.write(f'{epoch+1},{loss:.6f},{current_lr}\n')

    # except KeyboardInterrupt:
    #     print("\nTraining interrupted. Saving checkpoint...")

    # except Exception as e:
    #     print(f"\nERROR: {str(e)}")
    #     raise e

    # finally:
    #     print("\nTraining complete!")
    #     print(f"Best PSNR: {best_psnr:.2f}dB")
    #     print(f"Best SSIM: {best_ssim:.4f}")

    #     # Clean up
    #     if torch.cuda.is_available():
    #         torch.cuda.empty_cache()
    #     return model, losses, val_metric


In [3]:
class NKD_denoisers(nn.Module):

    def _extract_patch_vector(self, inp):
        batch, channel, _, _ = inp.shape
        patch_size = 2*self.patch_rad + 1
        t = torch.zeros(channel*patch_size*patch_size, channel, patch_size, patch_size, device=inp.device)
        count=0
        for c in range(channel):
            for i in range(patch_size):
                for j in range(patch_size):
                    t[count,c,i,j] = 1
                    count+=1

        return torch.functional.F.conv2d(inp, t)#.reshape(batch, -1, channel*patch_s*patch_s)

    # Defined in child class
    def _process_input(self):
        # to be defined in the child class
        pass

    # Defined in child class
    def _get_activation(self):
        # to be defined in the child class
        pass

    def forward(self, inp, sig=None):
        # print(torch.max(inp))
        # print(torch.min(inp))
        b, c, h, w = inp.shape

        guide = 2 * inp.clone() - 1
        b, _, h, w = guide.shape
        if self.blind:
            pass
        else:
            assert sig!=None, 'In the case of NON-blind model batch std. must be an input'
            guide = torch.cat(
                (guide, torch.ones((b,1,h,w), device=self.device, dtype=guide.dtype)*sig),
                dim = 1
            )

        # Extract guide weights
        # print("guide",guide.shape)
        guide = self._process_input(guide)
        # print("guide",guide.shape)
        guide_weights = self._get_activation(guide)
        # print("guide_weights",guide_weights)
        normalizing_factor = torch.sum(guide_weights, dim=1)
        # print("normalizing_factor",normalizing_factor)

        inp = inp[:, :, self.patch_rad:h-self.patch_rad, self.patch_rad:w-self.patch_rad].reshape(b,-1,self.in_channel)
        # print("inp",inp)
        out = torch.sum((inp*guide_weights), dim=1)/normalizing_factor
        # print("out",out)
        return out.reshape(b, self.in_channel)

    def train_step(self, train_loader, criterion, optimizer, lamda = 1e-3):
        self.train()
        avg_loss = 0
        count = 0

        for clean, noisy, sigma in train_loader:
            clean = clean.to(self.device, dtype=torch.float32)
            noisy = noisy.to(self.device, dtype=torch.float32)
            sigma = sigma.to(self.device, dtype=torch.float32)
            if len(noisy.shape)>4:
                c, h, w = noisy.size()[-3:]
                noisy= noisy.reshape(-1,c,h,w)

                if len(clean.shape)>2:
                    clean= clean.reshape(-1,c)

            # More efficient than zero_grad()
            optimizer.zero_grad(set_to_none=True)

            output = self(noisy, sigma)

            if lamda!=0:
                l1 = l1_loss(self)
            else:
                l1 = 0

            loss = criterion(output, clean)
            loss_t = loss + lamda*l1
            loss_t.backward()
            optimizer.step()

            b = output.size(0)
            count+=b
            avg_loss += (loss.item() * b)

        avg_loss = avg_loss/count
        return avg_loss

    def val_step(self, val_loader):
        self.eval()
        metric = {'psnr' : 0, 'ssim' : 0}

        with torch.no_grad():
            for clean, noisy, sigma in val_loader:
                clean = clean.to(self.device, dtype=torch.float32)
                noisy = noisy.to(self.device, dtype=torch.float32)
                sigma = sigma.to(self.device, dtype=torch.float32)
                b = clean.size(0)

                if len(noisy.shape)>4:
                    c, h, w = noisy.size()[-3:]
                    noisy= noisy.reshape(-1,c,h,w)
                
                if len(clean.shape)>2:
                    clean= clean.reshape(-1,c)
            
                if self.blind:
                    output = self(noisy)
                else:
                    output = self(noisy, sigma)

                metric['psnr'] += calculate_psnr(output, clean)
                metric['ssim'] += calculate_ssim(output, clean)

        metric['ssim'] = metric['ssim']/len(val_loader)
        metric['psnr'] = metric['psnr']/len(val_loader)
        return metric

class NKD_concat_kernel_learn(NKD_denoisers):
    def __init__(self, args, device): 
        super().__init__()
        self.device = device
        self.residual_depth = args.residual_depth
        self.proj_depth = args.proj_depth
        self.sigma = 1
        self.blind = args.blind
        self.latent_dim = args.latent_dim
        self.patch_rad = args.patch_rad
        patch_size = args.patch_rad * 2 + 1

        self.window_rad = args.window_rad
        self.in_channel = args.in_channel

        if args.blind:
            in_dim=2*args.in_channel*patch_size*patch_size
        else:
            in_dim=2*(args.in_channel+1)*patch_size*patch_size
        self.in_layer = nn.Linear(in_features=in_dim,out_features=args.latent_dim)
        self.in_residual = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_features=args.latent_dim,out_features=args.latent_dim*2),
                nn.LeakyReLU(),
                nn.Linear(in_features=args.latent_dim*2,out_features=args.latent_dim),
            )
            for _ in range(args.residual_depth)
        ])
        self.in_norm = nn.ModuleList([
            nn.LayerNorm(args.latent_dim)
            for _ in range(args.residual_depth)
        ])
        # self.in_proj = nn.Linear(in_features=in_dim,out_features=latent_dim)

        self.proj = nn.Sequential(*[
            nn.Sequential(
                nn.Linear(in_features=args.latent_dim//(2**i),out_features=args.latent_dim//(2**(i+1))),
                nn.LayerNorm(args.latent_dim//(2**(i+1))),
                nn.LeakyReLU(),
            )
            for i in range(args.proj_depth)
        ])
        self.out = nn.Linear(in_features=args.latent_dim//(2**(args.proj_depth)),out_features=args.in_channel)

    def _concat_patch_central_pixel(self, inp):
        b, c, h, w = inp.shape
        central_pixel = (inp[:,:,h//2,w//2])[:,:,None,None].expand_as(inp).permute(0,2,3,1).reshape(b,-1,c)
        return (torch.cat((inp.permute(0,2,3,1).reshape(b,-1,c),central_pixel), dim=-1), 
                torch.cat((central_pixel,inp.permute(0,2,3,1).reshape(b,-1,c)), dim=-1))

    def _get_activation(self, x):
        # print('xa', x.shape)
        x = self.in_layer(x)
        # print('xb', x.shape)
        for i in range(self.residual_depth):
            x = x + self.in_residual[i](x)
            x = self.in_norm[i](x)
            # print('xc', x)
        # x = self.in_proj(x)
        x = self.proj(x)
        # print('xd', x.shape)
        x = self.out(x)
        # print('xe', x)
        x1,x2 = torch.chunk(x, chunks=2, dim=1)
        # print('x1', x1)
        return torch.exp((x1+x2)/(-2))

    def _process_input(self, inp):
        # print("inp",inp.shape)
        x = self._extract_patch_vector(inp)
        # print("x",x.shape)
        x1, x2 = self._concat_patch_central_pixel(x)
        # print("x1",x1.shape)
        # print("x2",x2.shape)
        inp = torch.cat((x1,x2), dim=1)
        # print("inp",inp.shape)
        return inp

    def __str__(self):
        return '_concat_kl'+f'_{self.patch_rad}{self.window_rad}{self.in_channel}{self.latent_dim}_{self.residual_depth}{self.proj_depth}_{int(self.blind)}'

class NKD_diff_kernel_learn(NKD_denoisers):
    def __init__(self, args, device): 
        super().__init__()
        self.device = device
        self.residual_depth = args.residual_depth
        self.proj_depth = args.proj_depth
        self.sigma = 1
        self.blind = args.blind
        self.latent_dim = args.latent_dim
        self.patch_rad = args.patch_rad
        patch_size = args.patch_rad * 2 + 1

        self.window_rad = args.window_rad
        self.in_channel = args.in_channel

        if args.blind:
            in_dim=args.in_channel*patch_size*patch_size
        else:
            in_dim=(args.in_channel+1)*patch_size*patch_size

        self.in_layer = nn.Linear(in_features=in_dim,out_features=args.latent_dim)
        self.in_residual = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_features=args.latent_dim,out_features=args.latent_dim*2),
                nn.LeakyReLU(),
                nn.Linear(in_features=args.latent_dim*2,out_features=args.latent_dim),
            )
            for _ in range(args.residual_depth)
        ])
        self.in_norm = nn.ModuleList([
            nn.LayerNorm(args.latent_dim)
            for _ in range(args.residual_depth)
        ])
        # self.in_proj = nn.Linear(in_features=in_dim,out_features=latent_dim)

        self.proj = nn.Sequential(*[
            nn.Sequential(
                nn.Linear(in_features=args.latent_dim//(2**i),out_features=args.latent_dim//(2**(i+1))),
                nn.LayerNorm(args.latent_dim//(2**(i+1))),
                nn.LeakyReLU(),
            )
            for i in range(args.proj_depth)
        ])
        self.out = nn.Linear(in_features=args.latent_dim//(2**(args.proj_depth)),out_features=args.in_channel)

    def _diff_sq_central_pixel(self, inp):
        b, l, c = inp.shape
        central_pixel = (inp[:,l//2,:])[:,None,:].expand_as(inp)
        return (inp-central_pixel)**2

    def _get_activation(self, inp):
        x1 = self.in_layer(inp)
        for i in range(self.residual_depth):
            x1 = x1 + self.in_residual[i](x1)
            x1 = self.in_norm[i](x1)
        # x = self.in_proj(x)

        # print('x2', x1)
        x2 = self._diff_sq_central_pixel(x1)
        # print('x3', x2)
        x3 = self.proj(x2)
        x4 = self.out(x3)
        return torch.exp(x4/(-2))

    def _process_input(self, inp):
        # print('inp', inp)
        x = self._extract_patch_vector(inp)
        b, l, _, _ = x.shape
        # print('x', x)
        x = x.permute(0,2,3,1).reshape(b,-1,l)
        # print('x', x)
        return x

    def __str__(self):
        return 'diff_kl'+f'_{self.patch_rad}{self.window_rad}{self.in_channel}{self.latent_dim}_{self.residual_depth}{self.proj_depth}_{int(self.blind)}'

class NKD_rbf_feat_learn(NKD_denoisers):
    def __init__(self, args, device):
        super().__init__()
        self.device = device
        self.residual_depth = args.residual_depth
        self.proj_depth = args.proj_depth
        self.sigma = 1
        self.blind = args.blind
        self.latent_dim = args.latent_dim
        self.latent_channel_dim = args.latent_channel_dim
        self.patch_rad = args.patch_rad
        patch_size = args.patch_rad * 2 + 1

        self.window_rad = args.window_rad
        self.in_channel = args.in_channel

        if args.blind:
            in_dim=args.in_channel*patch_size*patch_size
        else:
            in_dim=(args.in_channel+1)*patch_size*patch_size

        self.in_dim=in_dim
        self.in_layer = nn.Linear(in_features=in_dim,out_features=args.latent_dim)
        self.in_residual = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_features=args.latent_dim,out_features=args.latent_dim*2),
                nn.LeakyReLU(),
                nn.Linear(in_features=args.latent_dim*2,out_features=args.latent_dim),
            )
            for _ in range(args.residual_depth)
        ])
        self.in_norm = nn.ModuleList([
            nn.LayerNorm(args.latent_dim)
            for _ in range(args.residual_depth)
        ])
        # self.in_proj = nn.Linear(in_features=in_dim,out_features=latent_dim)

        self.proj = nn.Sequential(*[
            nn.Sequential(
                nn.Linear(in_features=args.latent_dim//(2**i),out_features=args.latent_dim//(2**(i+1))),
                nn.LayerNorm(args.latent_dim//(2**(i+1))),
                nn.LeakyReLU(),
            )
            for i in range(args.proj_depth)
        ])
        self.out = nn.Linear(in_features=args.latent_dim//(2**(args.proj_depth)),out_features=args.in_channel*self.latent_channel_dim)

    def _diff_sq_patch_central_pixel(self, inp):
        b, n, c = inp.shape
        central_pixel = (inp[:,n//2,:])[:,None,:].expand_as(inp)
        return (inp-central_pixel)**2

    def _get_activation(self, x):
        x = self.in_layer(x)
        for i in range(self.residual_depth):
            x = x + self.in_residual[i](x)
            x = self.in_norm[i](x)
        # x = self.in_proj(x)

        # print('x1', x.shape)
        x = self.proj(x)

        # print('x2', x.shape)
        x = self.out(x)
        
        # print('x6', x.shape)
        out = self._diff_sq_patch_central_pixel(x)
        # print('out', out.shape)

        b, c, d = out.shape
        out = torch.sum(out.reshape(b, c, self.latent_channel_dim, self.in_channel), dim=-2)
        # print('out', out.shape)
        return torch.exp((out)/(-2))

    def _process_input(self, inp):
        b, c, _, _ = inp.shape
        # print('process_inp: o-_extract_patch_vector>',x)
        x = self._extract_patch_vector(inp).permute(0,2,3,1).reshape(b, -1, self.in_dim)
        # print('process_inp: _extract_patch_vector output>',x)
        return x

    def __str__(self):
        return 'rbf'+f'_{self.patch_rad}{self.window_rad}{self.in_channel}{self.latent_dim}{self.latent_channel_dim}_{self.residual_depth}{self.proj_depth}_{int(self.blind)}'


In [4]:
if __name__ == "__main__":
    # args.seed, args.in_folder, args.out_folder, 
    # args.color_mode:  'gray' or 'rgb', args.sigma_min, args.sigma_max, args.patch_size, 
    # args.epochs, args.batch_size, args.lr, args.sch: 'multistep', 'warmup_cosine' or 'cosine', args.lr_step, args.lr_gamma, 
    # args.val_freq, args.save_freq, args.progress_freq, args.save_img_freq,
    # args.blind, args.patch_rad, args.window_rad, args.in_channel, args.latent_dim, args.latent_channel_dim
    seed = 100
    torch.manual_seed(seed)
    np.random.seed(seed)
    class config:
        def __init__(self):
            pass

    args = config()

    # MODEL PARAMETERS
    args.architecture = NKD_rbf_feat_learn
    args.patch_rad = 1
    args.window_rad = 5
    args.in_channel = 1
    args.latent_dim = 1024
    args.latent_channel_dim = 64
    args.residual_depth = 2
    args.proj_depth = 2
    args.blind = True
    
    # TRAINING PARAMETERS
    args.in_folder = './data'
    args.out_folder = './output'
    args.color_mode = 'gray'
    args.sigma_min = 30
    args.sigma_max = 30
    args.req_patches = True
    args.patch_size = 128
    if args.req_patches:
        args.patch_size = args.window_rad+args.patch_rad
    else:
        pass
    args.epochs = 6000
    args.batch_size = 1
    args.lr = 3e-4
    args.sch = 'cosine'
    args.l1_reg_lamda = 0.00
    # #for warmup_cosine
    # args.num_warmup_steps = 200
    # args.num_cycles = 0.5
    # #for multistep
    # args.lr_step = None
    # args.lr_gamma = None

    # PROGRESS DATA PARAMETERS
    args.val_freq = 100
    args.save_freq = 200
    args.save_img = True
    args.save_img_freq = 200
    args.progress_freq = 1

    args.imgs_per_epoch = 1
    args.factor = None
    args.no_patches = 1024
    
    # RESUME TRAINING ?
    args.resume_training = False
    args.checkpoint_path = ''
    verbose = False
    train(args, verbose)


Loading training images...
Loaded 1 images from img_house
Total loaded 1 images from: img_house
Loaded 1 images for training


In [11]:
ad=NKD_rbf_feat_learn(args,'cpu')
print(str(ad))

rbf_feat_learn
